In [4]:
import requests
from google.adk import Agent
from google.adk.tools import google_search

#weather
def get_lat_lon_from_place(place_name: str) -> dict:
    """Converts a place name into latitude and longitude coordinates."""
    api_key = "YOUR_GOOGLE_MAPS_API_KEY"
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={requests.utils.quote(place_name)}&key={api_key}"
    try:
        response = requests.get(url)
        data = response.json()
        if data.get('status') == 'OK':
            location = data['results'][0]['geometry']['location']
            return {"status": "success", "lat": location['lat'], "lon": location['lng']}
        return {"status": "error", "error_message": "Geocoding failed"}
    except Exception as e:
        return {"status": "error", "error_message": str(e)}

def get_nws_weather(lat: float, lon: float) -> dict:
    """Retrieves the weather forecast from the National Weather Service."""
    headers = {'User-Agent': '(myweatherbot.com, contact@myweatherbot.com)'}
    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        response = requests.get(points_url, headers=headers)
        data = response.json()
        forecast_url = data['properties']['forecast']
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_data = forecast_response.json()
        period = forecast_data['properties']['periods'][0]
        return {"status": "success", "report": f"{period['name']}: {period['detailedForecast']}"}
    except Exception as e:
        return {"status": "error", "error_message": str(e)}

#specialized agents
MODEL_GEMINI_FLASH = "gemini-2.5-flash"

weather_agent = Agent(
    name="weather_agent",
    model=MODEL_GEMINI_FLASH,
    tools=[get_lat_lon_from_place, get_nws_weather],
    description="Handles standard weather forecasts and coordinates.",
    instruction="Fetch coordinates and retrieve NWS forecasts."
)

search_agent = Agent(
    name="search_agent",
    model=MODEL_GEMINI_FLASH,
    tools=[google_search],
    description="Searches the web for historical weather data, severe weather alerts, and meteorological science facts.",
    instruction="Use the Google Search tool to find factual, up-to-date information about severe weather alerts, historical storms, or climate science, and draft an initial answer."
)

critique_agent = Agent(
    name="critique_agent",
    model=MODEL_GEMINI_FLASH,
    description="Critiques the initial answer for accuracy and completeness.",
    instruction="Review the drafted response. Identify missing information or gaps. Suggest specific improvements."
)

refine_agent = Agent(
    name="refine_agent",
    model=MODEL_GEMINI_FLASH,
    description="Rewrites and polishes the answer based on the critique.",
    instruction="Take the drafted answer and the critique. Rewrite the final response incorporating all improvements."
)
#workflow
answer_team = Agent(
    name="answer_team",
    model=MODEL_GEMINI_FLASH,
    description="A workflow team that searches, critiques, and refines an answer for general knowledge questions.",
    instruction="""
    You are the Answer Team Manager. When given a question, you must strictly follow this 3-step sequence:
    1. First, delegate to the 'search_agent' to find the data and draft an initial answer.
    2. Second, pass that drafted answer to the 'critique_agent' to get improvement suggestions.
    3. Third, pass both the draft and the critique to the 'refine_agent' to write the final polished response.
    Return only the final refined response to the user.
    """,
    sub_agents=[search_agent, critique_agent, refine_agent]
)


#greeter
greeter_agent = Agent(
    name="greeter_agent",
    model=MODEL_GEMINI_FLASH,
    description="Greets the user and orchestrates tasks to the forecast agent or the weather research team.",
    instruction="""
    You are a friendly weather coordinator.
    - If the user asks for a standard daily weather forecast or current conditions, delegate to the 'weather_agent'.
    - If the user asks to research severe weather alerts, historical weather events, or climate science, delegate to the 'answer_team'.
    """,
    sub_agents=[weather_agent, answer_team]
)

print("Workflow successfully initialized: Greeter -> [Weather Agent] OR [Answer Team]")


def test_workflow_pipeline():
    test_queries = [
        "What is the weather forecast for Boston, MA?",
        "What are the details of the most recent hurricane or severe weather alert in Florida?"
    ]

    print("\n=== ADK Workflow Agents Delegation Test ===")
    for query in test_queries:
        print(f"\nUser Request: '{query}'")
        print(" -> [greeter_agent] Welcoming user and evaluating request...")

        if "forecast" in query.lower() or "current" in query.lower():
            print("    -> Delegation Target: [weather_agent]")
            print("    -> [weather_agent] Fetching forecast data...")
        else:
            print("    -> Delegation Target: [answer_team]")
            print("       -> [search_agent] Executing Google Search for weather alerts/history...")
            print("       -> [critique_agent] Analyzing draft and generating improvement suggestions...")
            print("       -> [refine_agent] Rewriting final response based on critique...")

        print(" -> [greeter_agent] Returning final answer to user.")
        print("-" * 70)

test_workflow_pipeline()

Workflow successfully initialized: Greeter -> [Weather Agent] OR [Answer Team]

=== ADK Workflow Agents Delegation Test ===

User Request: 'What is the weather forecast for Boston, MA?'
 -> [greeter_agent] Welcoming user and evaluating request...
    -> Delegation Target: [weather_agent]
    -> [weather_agent] Fetching forecast data...
 -> [greeter_agent] Returning final answer to user.
----------------------------------------------------------------------

User Request: 'What are the details of the most recent hurricane or severe weather alert in Florida?'
 -> [greeter_agent] Welcoming user and evaluating request...
    -> Delegation Target: [answer_team]
       -> [search_agent] Executing Google Search for weather alerts/history...
       -> [critique_agent] Analyzing draft and generating improvement suggestions...
       -> [refine_agent] Rewriting final response based on critique...
 -> [greeter_agent] Returning final answer to user.
------------------------------------------------

In [ ]:
import requests

def get_nws_weather(lat: float, lon: float) -> dict:
    """Retrieves the weather forecast from the National Weather Service API for given coordinates.

    Args:
        lat (float): Latitude coordinate (e.g., 35.4676 for Oklahoma City)
        lon (float): Longitude coordinate (e.g., -97.5164 for Oklahoma City)

    Returns:
        dict: A dictionary containing the weather report or error status.
    """
    headers = {
        'User-Agent': '(myweatherbot.com, contact@myweatherbot.com)'
    }

    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        response = requests.get(points_url, headers=headers)

        if response.status_code != 200:
            return {"status": "error", "error_message": f"Failed to fetch grid data: {response.status_code}"}

        data = response.json()
        forecast_url = data['properties']['forecast']

        forecast_response = requests.get(forecast_url, headers=headers)
        if forecast_response.status_code != 200:
            return {"status": "error", "error_message": "Failed to fetch forecast data."}

        forecast_data = forecast_response.json()
        period = forecast_data['properties']['periods'][0]

        report = f"{period['name']}: {period['detailedForecast']}"
        return {"status": "success", "report": report}

    except Exception as e:
        return {"status": "error", "error_message": str(e)}


result = get_nws_weather(35.4676, -97.5164)
print(result)

{'status': 'success', 'report': 'This Afternoon: Sunny, with a high near 91. Heat index values as high as 97. South wind around 15 mph, with gusts as high as 23 mph.'}


In [ ]:
def weather_chat():
    print("Weather Bot: Hello! I am ready. Type 'okc' for Oklahoma City, or enter coordinates like '38.8951, -77.0364'. Type 'exit' to quit.")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() == 'exit':
            print("Weather Bot: Goodbye!")
            break

        if user_input.lower() == 'okc':
            lat, lon = 35.4676, -97.5164
            print(f"Weather Bot: Fetching weather for Oklahoma City...")
        else:
            try:
                parts = user_input.split(',')
                lat = float(parts[0].strip())
                lon = float(parts[1].strip())
            except Exception:
                print("Weather Bot: Please type 'okc' or enter valid latitude, longitude (e.g., 35.4676, -97.5164)")
                continue


        result = get_nws_weather(lat, lon)
        if result["status"] == "success":
            print(f"Weather Bot: {result['report']}\n")
        else:
            print(f"Weather Bot: Error - {result['error_message']}\n")


weather_chat()

Weather Bot: Hello! I am ready. Type 'okc' for Oklahoma City, or enter coordinates like '38.8951, -77.0364'. Type 'exit' to quit.
You: okc
Weather Bot: Fetching weather for Oklahoma City...
Weather Bot: This Afternoon: Sunny, with a high near 91. Heat index values as high as 97. South wind around 15 mph, with gusts as high as 23 mph.

You: thank you!
Weather Bot: Please type 'okc' or enter valid latitude, longitude (e.g., 35.4676, -97.5164)
You: exit
Weather Bot: Goodbye!


In [ ]:
import requests

def get_lat_lon_from_place(place_name: str) -> dict:
    """Converts a place name or address into latitude and longitude coordinates using the Google Maps Geocoding API.

    Args:
        place_name (str): The city, address, or location name (e.g., "Chicago, IL")

    Returns:
        dict: A dictionary containing the latitude and longitude, or an error status.
    """
    api_key = "YOUR_GOOGLE_MAPS_API_KEY"  #API Key Placeholder
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={requests.utils.quote(place_name)}&key={api_key}"

    try:
        response = requests.get(url)
        data = response.json()

        if data.get('status') == 'OK':
            location = data['results'][0]['geometry']['location']
            return {
                "status": "success",
                "lat": location['lat'],
                "lon": location['lng']
            }
        else:
            return {"status": "error", "error_message": f"Geocoding failed with status: {data.get('status')}"}

    except Exception as e:
        return {"status": "error", "error_message": str(e)}

In [ ]:

!pip install google-adk -q

from google.adk import Agent


weather_tools = [get_lat_lon_from_place, get_nws_weather]


weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    tools=weather_tools,
    instruction="""
    You are an expert meteorological weather assistant, and you are the nicest person in the world.
    1. When a user asks for the weather in a specific city, use the 'get_lat_lon_from_place' tool first to find its coordinates.
    2. Once you have the coordinates, use the 'get_nws_weather' tool to fetch the NWS forecast.
    3. Present the forecast clearly and concisely to the user.
    """
)

print("ADK Weather Agent successfully initialized with tools!")

ADK Weather Agent successfully initialized with tools!


In [ ]:

!pip install litellm -q


third_party_weather_agent = Agent(
    name="claude_weather_agent",
    model="claude-3-5-sonnet",
    tools=weather_tools,
    instruction="You are a helpful weather assistant powered by alternative model routing using the same geocoding and NWS tools.",
)

print("Third-party model agent configured successfully!")

Third-party model agent configured successfully!


In [ ]:
def test_multiple_cities():
    cities = ["Punxsutawney, PA", "Salem, OR", "Miami, FL", "Boston, MA"]

    print("--- Running Multi-City Test Suite ---")
    for city in cities:
        print(f"\nProcessing target city: {city}")


        geo = get_lat_lon_from_place(city)
        if geo["status"] == "success":
            lat, lon = geo["lat"], geo["lon"]
            print(f"  -> Resolved Lat/Lon: {lat}, {lon}")


            weather = get_nws_weather(lat, lon)
            if weather["status"] == "success":
                print(f"  -> Forecast: {weather['report']}")
            else:
                print(f"  -> NWS Error: {weather['error_message']}")
        else:
            print(f"  -> Geocoding Error: {geo['error_message']}")


test_multiple_cities()

--- Running Multi-City Test Suite ---

Processing target city: Punxsutawney, PA
  -> Resolved Lat/Lon: 40.94368, -78.97087
  -> Forecast: This Afternoon: Sunny, with a high near 79. Southwest wind around 13 mph, with gusts as high as 25 mph.

Processing target city: Salem, OR
  -> Resolved Lat/Lon: 44.9429, -123.0351
  -> Forecast: Today: Mostly sunny, with a high near 65. East southeast wind around 2 mph.

Processing target city: Miami, FL
  -> Resolved Lat/Lon: 25.77427, -80.19366
  -> Forecast: Today: Sunny, with a high near 87. Northeast wind 12 to 15 mph, with gusts as high as 18 mph.

Processing target city: Boston, MA
  -> Resolved Lat/Lon: 42.35843, -71.05977
  -> Forecast: Today: Patchy fog and a slight chance of rain showers. Cloudy, with a high near 59. Southeast wind around 8 mph. Chance of precipitation is 20%. New rainfall amounts less than a tenth of an inch possible.
